# 03 — Fast-Path LightGBM and Legitimate-Novelty Models

Prepare features, train a transaction-risk model and a separate context model, then compare their outputs.

Run from the repository root with the project virtual environment selected.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from src.features.transaction_features import add_basic_transaction_features
from src.features.behavioural_features import add_behavioural_features
from src.features.context_features import add_context_features
from src.features.graph_features import add_peer_normalised_graph_features
from src.data.split_data import temporal_split
features = add_peer_normalised_graph_features(add_context_features(add_behavioural_features(add_basic_transaction_features(upi))))
train, val, test = temporal_split(features)

In [ ]:
from src.models.lightgbm_model import TransactionRiskModel
from src.models.context_model import LegitimateNoveltyModel
transaction_model = TransactionRiskModel.create()
transaction_model.fit(train, val)

In [ ]:
context_model = LegitimateNoveltyModel.create()
context_model.fit(train, val)

In [ ]:
sample = test[test['scenario'].isin(['auto_long_distance_legitimate','fake_refund_qr'])].head(10).copy()
sample['fraud_score'] = transaction_model.predict_proba(sample)
sample['legitimate_context_score'] = context_model.predict_proba(sample)
sample[['scenario','amount','fraud_score','legitimate_context_score']]